# 🎬 CineGraph — LightGCN Demo Notebook

**Paper**: LightGCN: Simplifying and Powering Graph Convolution Network for Recommendation  
**Authors**: He et al., SIGIR 2020  
**Dataset**: MovieLens-1M  

This notebook walks through the full pipeline:
1. Data loading & preprocessing
2. Graph construction
3. LightGCN training
4. Evaluation
5. Recommendations & LLM chat demo

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from src.data_loader import ML1MDataset
from src.graph import build_norm_adj
from src.model.lightgcn import LightGCN
from src.evaluate import evaluate, print_metrics

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Load & Explore MovieLens-1M

In [ ]:
dataset = ML1MDataset(data_dir='../data/ml-1m')

print(f'Users:      {dataset.n_users}')
print(f'Items:      {dataset.n_items}')
print(f'Train:      {len(dataset.train_df)}')
print(f'Val:        {len(dataset.val_df)}')
print(f'Test:       {len(dataset.test_df)}')

In [ ]:
# Interaction density
density = len(dataset.train_df) / (dataset.n_users * dataset.n_items) * 100
print(f'Graph density: {density:.4f}%')

# Distribution of interactions per user
user_counts = dataset.train_df.groupby('user_idx').size()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(user_counts, bins=50, color='#667eea', edgecolor='white', linewidth=0.5)
axes[0].set_title('Interactions per User (Train)')
axes[0].set_xlabel('# Interactions')
axes[0].set_ylabel('# Users')

item_counts = dataset.train_df.groupby('item_idx').size()
axes[1].hist(item_counts, bins=50, color='#764ba2', edgecolor='white', linewidth=0.5)
axes[1].set_title('Interactions per Item (Train)')
axes[1].set_xlabel('# Interactions')
axes[1].set_ylabel('# Items')

plt.tight_layout()
plt.show()

print(f'Avg interactions per user: {user_counts.mean():.1f}')
print(f'Avg interactions per item: {item_counts.mean():.1f}')

## 2. Build the Bipartite Graph

In [ ]:
from src.graph import build_interaction_matrix, build_adjacency_matrix, normalize_adjacency

R = build_interaction_matrix(dataset.train_df, dataset.n_users, dataset.n_items)
A = build_adjacency_matrix(R)

print(f'R (interaction matrix): {R.shape}  nnz={R.nnz}')
print(f'A (bipartite adj):      {A.shape}  nnz={A.nnz}')

norm_adj, _ = build_norm_adj(dataset.train_df, dataset.n_users, dataset.n_items, device)
print(f'Normalised adj (sparse tensor): {norm_adj.shape}')

## 3. Train LightGCN (Quick Run)

In [ ]:
from torch.optim import Adam

# Hyperparameters
EMB_DIM    = 64
N_LAYERS   = 3
LR         = 1e-3
LAMBDA_REG = 1e-4
BATCH_SIZE = 2048
EPOCHS     = 50   # quick demo — full training uses 200 epochs

model = LightGCN(
    n_users    = dataset.n_users,
    n_items    = dataset.n_items,
    emb_dim    = EMB_DIM,
    n_layers   = N_LAYERS,
    norm_adj   = norm_adj,
    lambda_reg = LAMBDA_REG,
).to(device)

print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

optimizer = Adam(model.parameters(), lr=LR)
val_dict  = dataset.get_val_negatives()
n_batches = max(1, len(dataset.train_df) // BATCH_SIZE)

train_losses, ndcg_vals = [], []

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0
    for _ in range(n_batches):
        u, pi, ni = dataset.sample_bpr_batch(BATCH_SIZE)
        u  = torch.LongTensor(u).to(device)
        pi = torch.LongTensor(pi).to(device)
        ni = torch.LongTensor(ni).to(device)
        bpr, reg = model(u, pi, ni)
        loss = bpr + LAMBDA_REG * reg
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        epoch_loss += loss.item()
    train_losses.append(epoch_loss / n_batches)

    if epoch % 10 == 0:
        val_m = evaluate(model, val_dict, dataset.n_users, dataset.n_items, device)
        ndcg_vals.append((epoch, val_m['NDCG@10']))
        print(f'Epoch {epoch:>3d}  loss={train_losses[-1]:.4f}  NDCG@10={val_m["NDCG@10"]:.4f}  HR@10={val_m["HR@10"]:.4f}')

In [ ]:
# Plot training curve
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(range(1, EPOCHS+1), train_losses, color='#667eea', linewidth=2)
ax1.set_title('Training Loss (BPR)')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.grid(alpha=0.3)

if ndcg_vals:
    epochs_v, ndcgs_v = zip(*ndcg_vals)
    ax2.plot(epochs_v, ndcgs_v, color='#764ba2', marker='o', linewidth=2)
    ax2.set_title('NDCG@10 on Validation Set')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('NDCG@10')
    ax2.grid(alpha=0.3)

plt.suptitle('LightGCN Training Progress', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Generate Recommendations

In [ ]:
USER_ID = 42

# Show user history
history = list(dataset.user_pos_items.get(USER_ID, set()))[:10]
print(f"=== User {USER_ID}'s Watch History ===")
for item_idx in history:
    print(f'  {dataset.get_movie_title(item_idx):50s}  [{dataset.get_movie_genres(item_idx)}]')

# Recommendations
exclude = dataset.user_all_pos.get(USER_ID, set())
recs = model.recommend(USER_ID, exclude_items=exclude, top_k=10)

print(f"\n=== Top 10 Recommendations for User {USER_ID} ===")
for rank, (item_idx, score) in enumerate(recs, 1):
    print(f'  #{rank:>2d}  {dataset.get_movie_title(item_idx):50s}  [{dataset.get_movie_genres(item_idx)}]  score={score:.4f}')

## 5. LLM Chat Demo

In [ ]:
import sys
sys.path.insert(0, '..')
from src.llm_agent import LLMAgent
from app.recommender_api import RecommenderAPI

# Load the full recommender API
rec_api = RecommenderAPI(checkpoint='../checkpoints/lightgcn_best.pt', data_dir='../data/ml-1m')
agent   = LLMAgent(recommender=rec_api)

# Chat!
queries = [
    "Hi! I'm user 42. What movies should I watch tonight?",
    "I love sci-fi movies, anything you recommend?",
    "Tell me more about the first movie you recommended.",
]

for q in queries:
    print(f"\n👤 USER: {q}")
    print(f"🤖 CINEGRAPH: {agent.chat(q)}")
    print('-' * 70)

## 6. Load Pre-trained Model & Full Evaluation

In [ ]:
import json, os

# Load results if available
results_files = {
    'LightGCN':       '../results/lightgcn_results.json',
    'LightGCN+SBERT': '../results/lightgcn_sbert_results.json',
    'Baselines':      '../results/baseline_results.json',
}

rows = []
for name, path in results_files.items():
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
        if name == 'Baselines':
            for model_name, d in data.items():
                rows.append({'Model': model_name.upper(), **d['test_metrics']})
        else:
            rows.append({'Model': name, **data['test_metrics']})

if rows:
    df = pd.DataFrame(rows).set_index('Model')
    display(df.style.highlight_max(axis=0).format('{:.4f}'))
else:
    print('No results yet — run training scripts first.')